In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
import scipy.stats as st
import statsmodels.api as sm 
import pylab as py 

# here are some of the tools we will use for our analyses
from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score


import itertools
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor


from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm


import pandas as pd
import csv
import sqlite3



In [2]:
#how does proximity to financial services, job density, population density, education level, and employment affect household income.

In [3]:
#Classification model starts here

In [4]:
income = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Household Income.csv", na_values="NA")
prox_bank = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Prox_bank_csv.csv", na_values="NA")
employment = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Employment.csv", na_values="NA")
job_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Job_Density_csv.csv", na_values="NA")
pop_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Population_Density_csv.csv", na_values="NA")
education = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Education Level - Bachelor's Degree.csv", na_values="NA")

In [5]:
income.columns

Index(['NPA', '2023'], dtype='object')

In [6]:
prox_bank.columns

Index(['NPA', '2023'], dtype='object')

In [7]:
employment.columns

Index(['NPA', '2023'], dtype='object')

In [8]:
job_density.columns

Index(['NPA', '2022'], dtype='object')

In [9]:
pop_density.columns

Index(['NPA', '2020'], dtype='object')

In [10]:
education.columns

Index(['NPA', '2023'], dtype='object')

In [11]:
conn = sqlite3.connect("my_data.db")

income.to_sql("income", conn, if_exists="replace", index=False)
prox_bank.to_sql("prox_bank", conn, if_exists="replace", index=False)
employment.to_sql("employment", conn, if_exists="replace", index=False)
job_density.to_sql("job_density", conn, if_exists="replace", index=False)
pop_density.to_sql("pop_density", conn, if_exists="replace", index=False)
education.to_sql("education", conn, if_exists="replace", index=False) 

459

In [12]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE income(
    NPA INT,
    median_income DOUBLE
)
""")

cursor.execute("""
CREATE TABLE prox_bank(
    NPA INT,
    "2023" DOUBLE
)
""")

cursor.execute("""
CREATE TABLE employment(
    NPA INT,
    employment_rate DOUBLE
)
""")

cursor.execute("""
CREATE TABLE job_density(
    NPA INT,
    "2022" INT
)
""")

cursor.execute("""
CREATE TABLE pop_density(
    NPA INT,
    "2020" INT
)
""")

cursor.execute("""
CREATE TABLE education(
    NPA INT,
    bachelors_percent DOUBLE
)
""")

In [13]:
for _, row in income.iterrows():
    for _, row in income.iterrows():
        cursor.execute("INSERT INTO income VALUES (?, ?)",
                       (row['NPA'], row['2023']))

In [14]:
for _, row in prox_bank.iterrows():
    for _, row in prox_bank.iterrows():
        cursor.execute("INSERT INTO prox_bank VALUES (?, ?)", 
                       (row['NPA'], row['2023']))

In [15]:
for _, row in employment.iterrows():
    cursor.execute("INSERT INTO employment VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

In [16]:
for _, row in job_density.iterrows():
    cursor.execute("INSERT INTO job_density VALUES (?, ?)", 
                   (row['NPA'], row['2022']))

In [17]:
for _, row in pop_density.iterrows():
    cursor.execute("INSERT INTO pop_density VALUES (?, ?)", 
                   (row['NPA'], row['2020']))

In [18]:
for _, row in education.iterrows():
    cursor.execute("INSERT INTO education VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

In [ ]:
cursor.execute("""
CREATE TABLE final_data AS
SELECT 
    i.NPA,
    i.median_income,
    pb."2023" as distance_to_bank,
    e.employment_rate,
    jd."2022" as job_density,
    pd."2020" as population_density,
    ed.bachelors_percent
FROM income i
FULL JOIN prox_bank pb ON i.NPA = pb.NPA
FULL JOIN employment e ON i.NPA = e.NPA
FULL JOIN job_density jd ON i.NPA = jd.NPA
FULL JOIN pop_density pd ON i.NPA = pd.NPA
FULL JOIN education ed ON i.NPA = ed.NPA
""")
conn.commit()

In [ ]:
cursor.execute("SELECT * FROM final_data")
final_data_results = cursor.fetchall()
final_data = pd.DataFrame(final_data_results, columns=['NPA', 'median_income', 'distance_to_bank', 'employment_rate', 'job_density', 'population_density', 'bachelors_percent'])
final_data